In [ ]:
from flask import Flask, request
import pandas as pd
import gspread
from oauth2client.service_account import ServiceAccountCredentials

app = Flask(__name__)

# --- Setup Google Sheets ---
scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
creds = ServiceAccountCredentials.from_json_keyfile_name("your_credentials.json", scope)
client = gspread.authorize(creds)
sheet = client.open("Guesty_Automation").sheet1

@app.route('/webhook', methods=['POST'])
def guesty_webhook():
    data = request.get_json()
    
    if data:
        # 1. Flatten the incoming data
        full_df = pd.json_normalize(data)
        
        # 2. SELECT ONLY THE COLUMNS YOU WANT
        # This list depends on Guesty's exact labels (e.g., 'guest.name', 'money.netAmount')
        keep_columns = [
            'reservation.confirmationCode', 
            'guest.fullName', 
            'listing.nickname', 
            'money.netEarnings', 
            'checkIn', 
            'checkOut'
        ]
        
        # 3. Create a filtered version (only keeping what exists in the data)
        filtered_df = full_df[full_df.columns.intersection(keep_columns)]
        
        # 4. Push to Google Sheets
        row_to_add = filtered_df.values.tolist()[0]
        sheet.append_row(row_to_add)
        
        return 'Success', 200
    return 'No data', 400

if __name__ == '__main__':
    app.run(port=5000)